# Lab 6: Examining Interaction Terms

In this lab, we will continue working with $\mathsf{R}$ to explore the concepts of interaction terms and partial $F$-tests. By the end of this lab, you will (hopefully)
have a clearer idea for how to visualize and interpret the interaction effects between predictors, to understand why individual $p$-values for main-effect coefficients can be misleading in the context of interactions, and to investigate the properties and the interpretations of partial $F$-tests for valid inference. 


*REMARK: Feel free to add cells throughout the lab if you would like to test things out or to have more intermediate steps. Any answers, though, should use the variable names listed in the question.*

*REMARK: Parts 1 and 2 of this lab contain material that might appear on Quiz \#2; Part 3 focuses on Lecture \#12, which is beyond the scope for what will be on Quiz \#2.*

In [ ]:
# Please install this initialization cell
library(ottr)
library(ggplot2)
library(dplyr)

## Part 1: Interpreting Models for Binary $\times$ Binary Interactions

We will continue with the running theme of sleep, which has featured in various lectures and labs and which will reappear here. Suppose a researcher is interested in studying how two common sleep-hygiene interventions affect a person's nightly sleep quality. A group of 40 participants were involved in the study, and these participants were randomly assigned to one of four conditions such that each subgroup contained exactly 10 participants. The two main interventions were: 

* *Blue light filter* (`blue_light`): whether one should use a blue-light-blocking screen filter after 9pm (the possible conditions were `"no"` or `"yes"`)
* *Caffeine cutoff* (`caffeine_cutoff`): whether one should avoid consuming any caffeine after 3pm (the possible conditions were `"no"` or `"yes"`)

At the end of a month-long period, each participant rated their average nightly sleep quality on a scale from 1 to 10, where 1 denoted terrible sleep quality and 10 denoted excellent sleep quality.

Run the cell below to generate and to preview the simulated data associated with this study. 

In [ ]:
set.seed(123) # DO NOT DELETE OR CHANGE THIS LINE!
n_per_group <- 10
sleep_df <- tibble(
  sleep_score = c(
    round(rnorm(n_per_group, mean = 5.0, sd = 1.2), 1),             # Neither intervention applied
    round(rnorm(n_per_group, mean = 6.1, sd = 1.2), 1),             # Blue light filter only
    round(rnorm(n_per_group, mean = 6.3, sd = 1.2), 1),             # Caffeine cutoff only
    round(pmin(rnorm(n_per_group, mean = 9.2, sd = 1.2), 10), 1)    # Both interventions applied
  ),
  blue_light = factor(
    c(rep("no", n_per_group), rep("yes", n_per_group),
      rep("no", n_per_group), rep("yes", n_per_group)),
    levels = c("no", "yes")
  ),
  caffeine_cutoff = factor(
    c(rep("no", 2 * n_per_group), rep("yes", 2 * n_per_group)),
    levels = c("no", "yes")
  )
)
head(sleep_df)

The following cell produces two different visualizations of the data: a grouped boxplot and an interaction line plot. These plots will help you answer some of the questions in this part of the lab. 

In [ ]:
# Grouped boxplot
p1 <- ggplot(sleep_df, aes(x = blue_light, y = sleep_score, fill = caffeine_cutoff)) +
        geom_boxplot() +
        labs(title = "Sleep Quality by Intervention Group", x = "Blue Light Filter",
            y = "Sleep Score", fill = "Caffeine Cutoff") + theme_bw()

# Interaction line plot of group means
group_means_plot <- sleep_df %>%
  group_by(blue_light, caffeine_cutoff) %>%
  summarize(mean_score = mean(sleep_score), .groups = "drop")

p2 <- ggplot(group_means_plot, aes(x = blue_light, y = mean_score,  group = caffeine_cutoff, color = caffeine_cutoff)) +
        geom_point(size = 3) + geom_line(linewidth = 1) +
        labs(title = "Interaction Plot: Mean Sleep Score", x = "Blue Light Filter",
            y = "Mean Sleep Score", color = "Caffeine Cutoff") + theme_bw()

print(p1)
print(p2)

**Question 1.1.** Using `dplyr`, compute the mean sleep score for each of the four treatment combinations and store the result in `group_means`, a dataframe with columns `blue_light`, `caffeine_cutoff`, and `mean_sleep_score` (in that order), with rows sorted as (no, no), (no, yes), (yes, no), and then (yes, yes).

*Hint: Your answer should take the form `group_by(...) %>% summarize(...) %>% arrange(...)`.*

In [ ]:
group_means <- NULL # YOUR CODE HERE
group_means

In [ ]:
. = ottr::check("tests/q1_1.R")

**Question 1.2.** Looking at `group_means`, we can compute the *difference-in-differences*, specifically with respect to caffeine, which we will denote by $\delta_\mathrm{cf}$. This is an informal measure indicating whether an interaction is present. To compute $\delta_\mathrm{cf}$, we need: 

* $\Delta_\mathrm{caffeine}$: the increase in mean sleep score from adding a blue light filter **among all participants who DID USE a caffeine cutoff**
* $\Delta_\mathrm{no\ caffeine}$: the increase in mean sleep score from adding a blue light filter **among all participants who DID NOT USE a caffeine cutoff**

Then, $\delta_\mathrm{cf} = \Delta_\mathrm{caffeine} - \Delta_\mathrm{no\ caffeine}$, which tells us by how much the caffeine cutoff boosted (if at all) the effectiveness of the blue light filter. Suppose there were no interaction between the interventions, meaning that the effect of the blue light filter was the same regardless of whether a subject used a caffeine cutoff. In this case, what would $\delta_\mathrm{cf}$ be equal to? 

1. It would be that $\delta_\mathrm{cf} < 0$ since the lack of interaction implies the two interventions work against each other.
2. It would be that $\delta_\mathrm{cf} = 1$ since the lack of interaction means the individual main effects must always sum to 1.
3. It would be that $\delta_\mathrm{cf} = 0$ since the lack of interaction means the individual $\Delta$ terms would be equal.
4. It would be that $\delta_\mathrm{cf}$ is the sum of both main effects since the lack of interaction makes these effects additive.

Assign your answer (1, 2, 3, or 4) to `no_interaction_delta` below.

In [ ]:
no_interaction_delta <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_2.R")

**Question 1.3.** Look at the interaction line plot from above. In that plot, the two lines connect the group means based on the presence of a caffeine cutoff (i.e., one line corresponds to `caffeine_cutoff = "no"` and the other to `caffeine_cutoff = "yes"`). What does it mean geometrically if the two lines are *not* parallel? 

1. The mean sleep scores for the two caffeine groups are equal. 
2. The two interventions (blue light filter, caffeine cutoff) are confounded and cannot be separated.
3. The blue light filter has no effect on sleep quality.
4. The effect of adding the blue light filter differs depending on whether the caffeine cutoff is put in place (i.e., an interaction is present). 

Assign your answer (1, 2, 3, or 4) to `non_parallel_interp`.

In [ ]:
non_parallel_interp <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_3.R")

Now, we wish to fit a full interaction model for this study, namely 

$$ \widehat{y} = \widehat{\beta}_0 + \widehat{\beta}_1x_1 + \widehat{\beta}_2x_2 + \widehat{\beta}_3x_1x_2,\tag{1}  $$

where $x_1$ corresponds to `blue_light` (0 = "no", 1 = "yes"), where $x_2$ corresponds to `caffeine_cutoff` (0 = "no", 1 = "yes"), and where $x_1x_2$ corresponds to their interaction term. 

**Question 1.4.** Fit the full interaction model using `lm()` and store the output in `sleep_model`. Then, extract the four coefficients ($\widehat{\beta}_0, \widehat{\beta}_1, \widehat{\beta}_2,$ and $\widehat{\beta}_3$) and store them in a numeric vector `sleep_coeffs`. The names in `sleep_coeffs` should be exactly as they appear in the `summary()` output of your model (e.g. you should have a `"(Intercept)"`, `"blue_lightyes"`, etc.).

*Hint: Your solution for `sleep_model` should take a similar form to the `lm()` expression in Slide 15 of Lecture \#11.*

*Hint: The function `coef()` might be useful.*

In [ ]:
# Fit linear model
sleep_model <- NULL # YOUR CODE HERE
# Extract model coefficients 
sleep_coeffs <- NULL # YOUR CODE HERE

# Observe summary (check to match names/values of coefficients)
summary(sleep_model)

In [ ]:
. = ottr::check("tests/q1_4.R")

**Question 1.5.1.** The coefficient $\widehat{\beta}_1$ corresponds to `blue_lightyes` in the model output. Which of the following is the most correct interpretation of this coefficient?

1. The estimated increase in the mean sleep score associated with using a blue light filter among participants who did not use a caffeine cutoff.
2. The estimated increase in the mean sleep score associated with using a blue light filter among participants who did use a caffeine cutoff.
3. The estimated average effect of the blue light filter, having been averaged over both caffeine groups.
4. The estimated difference in the sleep scores between the two caffeine groups, having kept the blue light filter constant across the groups.

Assign your answer (1, 2, 3, or 4) to `beta1_interpretation`. 

In [ ]:
beta1_interpretation <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_5_1.R")

**Question 1.5.2.** The coefficient $\widehat{\beta}_3$ corresponds to the interaction term (`blue_lightyes:caffeine_cutoffyes`) in the model output. Which of the following is the most correct interpretation of this coefficient?

1. The estimated effect of the blue light filter, having been averaged over both caffeine groups.
2. The estimated effect of using both interventions simultaneously, over and above either of the interventions on their own.
3. The estimated difference in the effect of the blue light filter between participants who used a caffeine cutoff and those who did not.
4. Answer choices 2 and 3 are both correct ways of describing the same quantity.

Assign your answer (1, 2, 3, or 4) to `beta3_interpretation`. 

In [ ]:
beta3_interpretation <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_5_2.R")

**Question 1.5.3.** The coefficient $\widehat{\beta}_0$ corresponds to the intercept in the model output. Which of the following is the most correct interpretation of this coefficient?

1. The average sleep score for every participant in the entire study.
2. The average sleep score for a participant who used neither a blue light filter nor a caffeine cutoff.
3. The average sleep score for a participant who used both a blue light filter and a caffeine cutoff. 
4. None of the above interpretations are correct for $\widehat{\beta}_0$.

Assign your answer (1, 2, 3, or 4) to `beta0_interpretation`. 

In [ ]:
beta0_interpretation <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_5_3.R")

**Question 1.6.** Equation (1) captures our fitted model for this particular study. Based on this equation, compute the estimated effect of using the blue light filter among participants who did use the caffeine cutoff. That is, find the predicted increase in sleep score when using a blue light filter, having set $x_2 = 1$. Store this estimated effect in `bl_effect_with_cc`.

In [ ]:
bl_effect_with_cc <- NULL # YOUR CODE HERE
bl_effect_with_cc

In [ ]:
. = ottr::check("tests/q1_6.R")

Notice that your answer to **Question 1.6** should match, numerically at least, with the quantity $\Delta_\mathrm{caffeine}$ introduced in **Question 1.2** above. While that question was focusing on the value for $\delta_\mathrm{cf}$ specifically, the two quantities are related. Take a second to think about why `bl_effect_with_cc` should be equal to $\Delta_\mathrm{caffeine}$.

**Question 1.7.** Suppose you calculated $\widehat{y}_i$ for a specific participant (the predicted value of our fitted model for this participant) and subtracted it from their actual recorded sleep score $y_i$. This difference is the *residual*: $ e_i \coloneqq y_i - \widehat{y}_i.$

Under the assumption that the interaction model described in Equation (1) is the "perfect fit" for the group means, what would the sum of the residuals <u>within each of the four groups</u> be? 

1. The sum would be exactly equal to 1.2 since this was the standard deviation that was used to generate the simulated study.
2. The sum would be exactly equal to 10 since this was the maximum possible score any participant could have assigned to their nightly sleep quality.
3. The sum would be exactly equal to 0 since the ordinary least-squares regression line (or, group means, in this case) passes exactly through the average of each subgroup.
4. The sum would be exactly equal to $\widehat{\beta}_3$ since this coefficient represents the "leftover" effect in our model. 

Assign your answer (1, 2, 3, or 4) to `residual_sum_interp`. 


In [ ]:
residual_sum_interp <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_7.R")

## Part 2: A Closer Look at $F$-tests and $p$-values

In this part of the lab, we'll explore the specific output of our fitted model from earlier. We'll explore why the main effect $p$-value is not always of interest (in isolation) and we'll try to assess how a predictor contributes to the model via partial $F$-tests. 

Run the following cell to refamiliarize yourself with the produed output of our fitted model from earlier.

In [ ]:
# Run this cell -- do not modify!
summary(sleep_model)

**Question 2.1.** From the output just produced, look at the $F$-statistic and its corresponding $p$-value. Which statement best describes the null hypothesis this overall $F$-test is evaluating?

1. $H_0: \beta_0 = \beta_1 = \beta_2 = \beta_3 = 0$, which is to say all of the model coefficients are equal to 0.
2. $H_0: \beta_1 = \beta_2 = \beta_3 = 0$, which is to say all of the predictors have no association with sleep quality.
3. $H_0: \mu_1 = \mu_2 = \mu_3 = \mu_4$, which is to say all of the four group means are the same.
4. $H_0: \beta_3 = 0$, which is to say there is no interaction between the two predictors.
5. $H_0: \beta_1 = \beta_2 \neq \beta_0, \beta_3$, which is to say the two predictors have the same association with sleep quality on their own but the intercept and interaction differ. 

Select all that are correct and store your choice(s) as a vector into `overall_ftest` below. If multiple options are correct, put them in the vector in ascending order.

In [ ]:
overall_ftest <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_1.R")

**Question 2.2.** The individual $p$-value for `blue_lightyes` in the summary table provided may differ substantially from the partial $F$-test $p$-value you will compute later. (This observation is similar to the one we saw in class, for the other example obviously.) Why is the individual $p$-value for $\widehat{\beta}_1$ not by itself enough to explain whether the blue light filter actually matters for sleep quality? 

1. The $p$-value for $\widehat{\beta}_1$ and the partial $F$-test would always give the same answer when exactly one term is being tested, so there is no real difference.
2. The $p$-value for $\widehat{\beta}_1$ would be inflated because it does not account for multiple comparisons across the four coefficients. 
3. The individual $p$-value for $\widehat{\beta}_1$ tests $H_0: \beta_1 = 0$, which only evaluates the blue filter's effect when the caffeine cutoff is not used (i.e., $x_2 = 0$). Should the blue filter only contribute when combined with the cutoff, this $p$-value will be large even though the filter genuinely contributes.
4. The individual $p$-values in multiple linear regression are always unreliable and biased, meaning they should not be used in practice. 

Assign your answer (1, 2, 3, or 4) to `main_pval_limitation`.

In [ ]:
main_pval_limitation <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_2.R")

The key insight from this last question has to do with the fact that individual $p$-values for specific coefficients are designed to test specific null hypotheses, namely hypotheses intended to test whether one intervention has an effect while the other is ignored (i.e., $x_k = 0$ for one of the interventions studied). This can lead to misleading conclusions when an interaction is present. 

To make this more concrete, suppose we compared the following simulated scenarios: 
- *Scenario A*: The blue light filter has a genuine main effect ($\beta_1 = 2$, say) while there <u>is no interaction</u> (i.e., $\beta_3 = 0$). Here, the main effect $p$-value and the partial $F$-test should agree.
- *Scenario B*: The blue light filter has essentially no effect on its own (i.e., $\beta_1 \approx 0$) but exhibits <u>a strong interaction</u> with the caffeine cutoff ($\beta_3 = 3$, say). The filter only contributes meaningfully when combined with the caffeine cutoff, meaning that the main effect $p$-value would be misleading here. 

Run the cell below to generate both scenario datasets. (In these scenarios, suppose we can simulate a number of participants that was infeasible for the study itself.)

In [ ]:
# Run this cell -- do not modify!
set.seed(152)
n_s <- 50                                   # Number of participants per group for scenarios only

# Create shared design matrix across scenarios
x1 <- c(rep(0, 2 * n_s), rep(1, 2 * n_s))                       # blue_light (encoded as 0/1 for "no"/"yes")
x2 <- c(rep(0, n_s), rep(1, n_s), rep(0, n_s), rep(1, n_s))     # caffeine_cutoff (encoded as 0/1 for "no"/"yes")

# Scenario A: main effect of beta_1 = 2 but no interaction
y_A <- 3 + 2 * x1 + 2 * x2 + 0 * x1 * x2 + rnorm(4 * n_s, 0, 2)
scen_A <- data.frame(y = y_A, x1 = factor(x1), x2 = factor(x2))

# Scenario B: main effect of x1 ≈ 0, STRONG interaction (beta3 = 3)
y_B <- 3 + 0.1 * x1 + 2 * x2 + 3 * x1 * x2 + rnorm(4 * n_s, 0, 2)
scen_B <- data.frame(y = y_B, x1 = factor(x1), x2 = factor(x2))

cat("Scenario A group means:\n")
print(aggregate(y ~ x1 + x2, data = scen_A, FUN = mean))
cat("\nScenario B group means:\n")
print(aggregate(y ~ x1 + x2, data = scen_B, FUN = mean))

For reference, it may be helpful to visualize both scenarios with interaction plots similar to those in Part 1. If you would want to do this, add cells here to generate those plots.

**Question 2.3.1.** Fit the full interaction model using the *Scenario A* data and store the model in `lm_scen_A`. Obtain the $p$-value for the `x1` coefficient (essentially the individual $t$-test for $\widehat{\beta}_1$) and store it in `pval_x1_A`.

*Hint: The line `summary(model)$coefficients` produces a matrix that includes coefficient estimates, standard errors, $t$-values and $p$-values.*

In [ ]:
lm_scen_A <- NULL # YOUR CODE HERE
pval_x1_A <- NULL # YOUR CODE HERE
pval_x1_A

In [ ]:
. = ottr::check("tests/q2_3_1.R")

**Question 2.3.2.** Now fit the full interaction model using the *Scenario B* data and store the model in `lm_scen_B`. Obtain the $p$-value for the `x1` coefficient (essentially the individual $t$-test for $\widehat{\beta}_1$) and store it in `pval_x1_B`.

*Hint: The line `summary(model)$coefficients` produces a matrix that includes coefficient estimates, standard errors, $t$-values and $p$-values.*

In [ ]:
lm_scen_B <- NULL # YOUR CODE HERE
pval_x1_B <- NULL # YOUR CODE HERE
pval_x1_B

In [ ]:
. = ottr::check("tests/q2_3_2.R")

**Question 2.4.1.** In *Scenario B*, even though the individual $p$-value for `x1` is quite large, the blue light filter does have a real effect, but only when $x_2 = 1$. Which <u>single</u> $p$-value in `summary(lm_scen_B)` reveals most clearly that the blue light filter still matters for sleep quality? 

1. The $p$-value for `x1:x2` (the interaction term), which will be small because it is picking up the added benefit of the filter when combined with the caffeine cutoff.
2. The $p$-value for `x2` (the caffeine cutoff main effect), which will be small because it indicates this intervention benefits sleep quality.
3. The $p$-value for the intercept, which will be small because it indicates the control group (no blue light filter, no caffeine cutoff) has a sleep quality score that is non-zero.
4. None of the individual $t$-test $p$-values, which means that we can only capture this properly via a partial $F$-test.  

Assign your answer (1, 2, 3, or 4) to `relevant_pvals_B` below.



In [ ]:
relevant_pvals_B <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_4_1.R")

**Question 2.4.2.** To summarize, the main limitation exposed by *Scenario B* is that the individual $p$-value for $\widehat{\beta}_1$ is the $p$-value for the test 

$$H_0: \beta_1 = 0\qquad\text{vs.}\qquad H_A: \beta_1 \neq 0.$$ 

In a model that also contains terms $x_2$ and $x_1x_2$, this null hypothesis would suggest $x_1$ has no effect **specifically when** $x_2 = 0$. As such, choose whichever answer choice best finishes this statement: 


$$ \text{The fact that the individual $p$-value for $\widehat{\beta}_1$ can yield a misleading conclusion is because ....}$$

1. ...the $p$-value for $\widehat{\beta}_1$ depends on the sample size of each group rather than the true effect size. Concluding that $x_1$ does not matter based on this individual $p$-value ignores the fact that a large $p$-value may simply reflect a lack of statistical power to detect a non-zero effect.
2. ...the individual $t$-test for $\widehat{\beta}_1$ is only valid when no interaction term is included in our fitted model. Concluding that $x_1$ does not matter based on this individual $p$-value ignores the mathematical reality that adding an interaction term moves us from a "general average effect" to a "specific conditional effect".
3. ...the regression coefficients are always biased when an interaction between variables is present. Concluding that $x_1$ does not matter based on this individual $p$-value ignores the possibility that the coefficient estimate $\widehat{\beta}_1$ could be skewed by other variables in our model.
4. ...even if $\beta_1 = 0$, the predictor $x_1$ could still have a non-trivial effect on $y$ when $x_2 \neq 0$ via the interaction term. Concluding that $x_1$ does not matter based on this individual $p$-value ignores the possibility that the interaction term could produce a non-trivial effect.

Assign your answer (1, 2, 3, or 4) to `main_effect_pval`.

In [ ]:
main_effect_pval <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_4_2.R")

As mentioned, we often need to be careful about the individual $p$-values in these interaction contexts. We discussed in lecture how the correct way to assess whether a predictor (including all terms that involve it) contributes to a model is with a **partial $F$-test**, comparing the full model to a reduced model (our null) that omits all terms containing the particular predictor. 

We return to our original data, `sleep_df`, and the model we fitted, `sleep_model`. 

In [ ]:
# Run this cell -- do not modify!
summary(sleep_model)

**Question 2.5.1.** Using the sleep data, test whether the blue light filter contributes to the model **at all**. That is, test $H_0: \beta_1 = \beta_3 = 0$, which is to say we want to drop both the `blue_light` main effect and the interaction between it and the caffeine cutoff. 

Fit the appropriate null model `no_bl_null_model` and then run `anova(no_bl_null_model, sleep_model)`. Store the resulting $p$-value in `pval_bl_joint`.

*Hint: The null model should only consider `caffeine_cutoff` as a predictor. It may also be helpful to save your `anova()` output.*

In [ ]:
no_bl_null_model <- NULL # YOUR CODE HERE
pval_bl_joint <- NULL # YOUR CODE HERE

# Check null model
summary(no_bl_null_model)


In [ ]:
. = ottr::check("tests/q2_5_1.R")

**Question 2.5.2.** The partial $F$-test in **Question 2.5.1** drops two terms ($\beta_1, \beta_3$) from our fitted model. How many are the degrees of freedom in the numerator for our partial $F$-test?

1. 1, because we had two predictors (interventions) for our model.
2. 2, because we are dropping two of the four coefficients from our model.
3. 3, because the full model has three non-intercept terms in it.
4. 28, because the study had 32 subjects and there are four coefficients.

Assign your answer (1, 2, 3, or 4) to `ftest_df_num`. 

*Hint: Check the `Df` column in the `anova()` output from the previous question.*

In [ ]:
ftest_df_num <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_5_2.R")

**Question 2.5.3.** Now test whether the caffeine cutoff contributes to the model **at all**. That is, test $H_0: \beta_2 = \beta_3 = 0$, which is to say we want to drop both the `caffeine_cutoff` main effect and the interaction between it and the blue light filter.

Fit the appropriate null model `no_cc_null_model` and then run `anova(no_cc_null_model, sleep_model)`. Store the resulting $p$-value in `pval_cc_joint`.

*Hint: Your code should be very similar to the one you wrote for **Question 2.5.1**.*

In [ ]:
no_cc_null_model <- NULL # YOUR CODE HERE
pval_cc_joint <- NULL # YOUR CODE HERE

# Check null model
summary(no_cc_null_model)


In [ ]:
. = ottr::check("tests/q2_5_3.R")

**Question 2.6.1.** Test whether there is an interaction — $H_0: \beta_3 = 0$ — by comparing `sleep_model` to a null model that contains only the two main effects. Store the $p$-value of this comparison in `pval_no_interaction`.

In [ ]:
no_interaction_null_model <- NULL # YOUR CODE HERE
pval_no_interaction <- NULL # YOUR CODE HERE
pval_no_interaction

In [ ]:
. = ottr::check("tests/q2_6_1.R")

**Question 2.6.2.** In the previous question, we dropped one coefficient ($\beta_3$) from our fitted model. Compare `pval_no_interaction` to the individual $p$-value for the interaction term in `summary(sleep_model)`. Which statement is correct? 

1. The two $p$-values test different hypotheses and, thus, will always differ.
2. The partial $F$-test $p$-value is always smaller than the individual $p$-value since it accounts for the correlation between predictors.
3. The partial $F$-test is only equivalent to the individual test when the sample size is very large, which is not the case here.
4. The two $p$-values are always equal when exactly one term is dropped because the partial $F$-test with one degrees of freedom in the numerator is equivalent to the two-sided $t$-test on the same coefficient (i.e., $F = t^2$.)

Assign your answer (1, 2, 3, or 4) to `ftest_vs_ttest`.

In [ ]:
ftest_vs_ttest <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_6_2.R")

**Question 2.5** and **Question 2.6** show that the individual $t$-test for $\widehat{\beta}_1$ and the partial $F$-test for $H_0: \beta_1 = \beta_3 = 0$ yielded different results. While the differing results did not suggest different decisions with respect to their corresponding null hypotheses, this may not be the case in other examples.

We can more easily quantify this difference in results via simulations by explicitly comparing the power of the two tests. To do this, we will still use our $2 \times 2$ factorial design with $n$ participants per subgroup ($4n$ total participants). Our true model is:

$$ y = 3 + \beta_1x_{1} + \beta_2x_{2} + \beta_3x_{1}x_{2} + \varepsilon, \quad\text{with $\varepsilon \sim \mathcal{N}(0, \sigma^2)$}. \tag{2}$$

For these simulations, we will fix $\beta_2 = 1$ and $\sigma = 2.5$, but $\beta_1$ and $\beta_3$ will vary across scenarios. 

**Question 2.7.** Complete the `sim_ftest_power` function below. For each of `n_sim` replications, your function should: 

1. Create a design matrix for $x_1, x_2$. 
2. Simulate some data from the true model for $y$ described in equation (2) with the given `beta1, beta3`, and with the fixed `beta2 = 1, sigma=2.5` and with `n_per_group` observations per subgroup.
2. Fit the full model, which includes the interaction term. 
3. Fit the null model, which omits `x1` and its interaction.
4. Record whether the individual $t$-test for `x1` in the full model rejects its null hypothesis at the significance level `alpha`.
5. Record whether the partial $F$-test from `anova(null_model, full_model)` rejects its null hypothesis at the significance level `alpha`.

Your function should return a numeric vector with entries `ttest_rej`, corresponding to the replications of the $t$-tests that were rejected, and `ftest_rej`, corresponding to the replications of the partial $F$-tests that were rejected. 

*Note: In the design matrix, make sure that `x1, x2` are numeric vectors encoded as 0s and 1s, not as factors, so that `summary()$coefficients` has the precise row name `"x1"`.*

In [ ]:
sim_ftest_power <- function(n_per_group, beta1, beta2, beta3, sigma = 2.5, n_sim = 500, alpha = 0.05) {
    # YOUR CODE HERE
}

# Test your function (if implemented correctly, rejection rates should be near alpha = 0.05)
set.seed(123) # DO NOT DELETE OR CHANGE THIS LINE!
sanity_check_ftest <- sim_ftest_power(n_per_group = 25, beta1 = 0, beta2 = 1, beta3 = 0, sigma = 2.5, n_sim = 500, alpha = 0.05)
sanity_check_ftest

In [ ]:
. = ottr::check("tests/q2_7.R")

Run the following cell to evaluate your function under a couple of scenarios and to produce a summary plot. 

*Note: This cell might take a minute or two to run.*

In [ ]:
set.seed(123) # DO NOT DELETE OR CHANGE THIS LINE!

# Scenario 1: No interaction, main effect of x1 only
result_scen1 <- sim_ftest_power(n_per_group = 50, beta1 = 2, beta2 = 1, beta3 = 0, sigma = 2.5, n_sim = 1000, alpha = 0.05)
# Scenario 2: Mixed scenario with moderate main effect and interaction
result_scen2 <- sim_ftest_power(n_per_group = 50, beta1 = 1.2, beta2 = 1, beta3 = 1.5, sigma = 2.5, n_sim = 1000, alpha = 0.05)
# Scenario 3: No main effect of x1, strong interaction
result_scen3 <- sim_ftest_power(n_per_group = 50, beta1 = 0, beta2 = 1, beta3 = 3, sigma = 2.5, n_sim = 1000, alpha = 0.05)

# Build summary table
sim_results_combined <- data.frame(
    Description = rep(c("Main Effect, No Interaction",  "Mixed Main Effect and Interaction", "No Main Effect but Interaction"), each = 2),
    Scenario = rep(c("1: beta1=2, beta3=0", "2: beta1=1.2, beta3=1.5", "3: beta1=0, beta3=3"), each=2),
    Test = c("Individual t-test", "Partial F-test", "Individual t-test", "Partial F-test", "Individual t-test", "Partial F-test"),
    Rej_Rate = c(result_scen1["ttest_rej"], result_scen1["ftest_rej"], result_scen2["ttest_rej"], result_scen2["ftest_rej"], 
                result_scen3["ttest_rej"], result_scen3["ftest_rej"])
)
print(sim_results_combined)

# Interpretation of results
par(bg = "white", mar = c(8, 4, 4, 2))
bar_vals <- c(result_scen1["ttest_rej"], result_scen1["ftest_rej"],
              result_scen2["ttest_rej"], result_scen2["ftest_rej"],
              result_scen3["ttest_rej"], result_scen3["ftest_rej"])
bar_names <- c("Scen1\nt-test", "Scen1\nF-test",
               "Scen2\nt-test", "Scen2\nF-test",
               "Scen3\nt-test", "Scen3\nF-test")
bp <- barplot(bar_vals, names.arg = bar_names, ylim = c(0, 1),
              col = rep(c("steelblue", "coral"), 3),
              ylab = "Rejection Rate",
              main = "Individual t-test vs Partial F-test Power")

abline(h = 0.05, lty = 2, col = "darkgrey")
legend("topright", fill = c("steelblue", "coral"),
       legend = c("Individual t-test", "Partial F-test"))


**Question 2.8.** Using the scenarios from above, store the rejection rate of the individual $t$-tests for Scenario 1 ($\beta_1 = 2, \beta_3 = 0$) and for Scenario 3 ($\beta_1 = 0, \beta_3 = 3$) in `ttest_rej_scen1` and `ttest_rej_scen3`, respectively.

Then, store the rejection rate of the partial $F$-tests for Scenario 1 ($\beta_1 = 2, \beta_3 = 0$) and for Scenario 3 ($\beta_1 = 0, \beta_3 = 3$) in `ftest_rej_scen1` and `ftest_rej_scen3`, respectively.

In [ ]:
ttest_rej_scen1 <- NULL # YOUR CODE HERE
ttest_rej_scen3 <- NULL # YOUR CODE HERE
ftest_rej_scen1 <- NULL # YOUR CODE HERE
ftest_rej_scen3 <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_8.R")

**Question 2.9.** A researcher is reviewing the power simulation results for the sleep study. Based on the patterns observed across the three simulated scenarios and in the bar plot above, which of the following statements provides the most insightful summary between the individual $t$-tests for $\widehat{\beta}_1$ and the partial $F$-test for the blue light filter? 

1. The partial $F$-test is generally superior to the individual $t$-test because it always achieves a higher rejection rate, regardless of whether an interaction is present or not.
2. In Scenario 1, the high rejection rates for both tests proves that the two different tests are statistically redundant when no interaction is present between the interventions; this means practitioners should always prefer the simpler $t$-test to avoid the complexity of a partial $F$-test.
3. In Scenario 3, the individual $t$-test fails to detect the effect of the blue light filter (rejecting only at the $\alpha$ level) because it evaluates the filter's impact only when the caffeine cutoff is absent; the partial $F$-test correctly identifies the filter's significance by evaluating both the main effect and the interaction simultaneously.
4. The individual $t$-test is a more precise measure of "pure" intervention effects, whereas the partial $F$-test is easily "fooled" into a conclusion of significance by the presence of the caffeine cutoff even when the blue light filter itself does not have a main effect on sleep quality.

Assign your answer (1, 2, 3, or 4) to `power_comparison`. 

*Hint: More than one option may be defensible, but choose the answer choice which best (most accurately) describes the difference the tests.*


In [ ]:
power_comparison <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_9.R")

#### This is the end of the material covered in Quiz \#2 (Lectures 6 through 11).

## Part 3: Continuous $\times$ Binary Interactions

So far, we have focused on two binary predictors, but interaction terms can also appear when one of the predictors is continuous. In the case that one is continuous and the other binary, which is the setting we focus on here, the interaction allows the slope of the continuous predictor to differ between groups – not just the intercept, a change from the previous setting. 

A dietician measured the resting heart rate (in beats per minute, or bpm) of 200 adults along with both their average daily step count and whether they followed a calorie-restricted diet. In this example, we have that the daily step count is a continuous predictor while the presence of a diet remains a binary predictor. The dietician was interested in seeing how these two predictors affected the adults' resting heart rates, with the premise that a lower resting heart rate means the person is healthier. 

Run the cell below to generate the preview the simulated data. 

In [ ]:
# Run this cell -- do not modify!
set.seed(123)
n_heart <- 200
daily_steps <- round(runif(n_heart, min = 2000, max = 15000))
on_diet <- sample(c("no", "yes"), n_heart, replace = TRUE)
heart_rate <- ifelse(
  on_diet == "no",
  91 - 0.003 * daily_steps,
  78 - 0.005  * daily_steps
) + rnorm(n_heart, mean = 0, sd = 3)
heart_rate <- round(heart_rate, 1)
heart_df <- data.frame(
  heart_rate  = heart_rate,
  daily_steps = daily_steps,
  on_diet     = factor(on_diet, levels = c("no", "yes"))
)
head(heart_df)

Run the following cell to produce a scatterplot with separate regression lines fitted to each diet group. Look out for whether or not the lines are parallel.

In [ ]:
# Run this cell -- do not modify!
ggplot(heart_df, aes(x = daily_steps, y = heart_rate, color = on_diet, shape = on_diet)) +
  geom_point(alpha = 0.6) +
  geom_smooth(method = "lm", se = FALSE, linewidth = 1.2) +
  labs(title = "Resting Heart Rate vs. Daily Steps by Diet Group", x = "Average Daily Steps",
        y = "Resting Heart Rate (bpm)", color = "On Diet", shape = "On Diet") +
  theme_bw()

**Question 3.1.** Fit a linear regression model of `heart_rate` on `daily_steps` and `on_diet` assuming there is no interaction present between the two (i.e., fit a model without an interaction term). Store the model in `hr_model_no_interaction` and extract the coefficient for `on_dietyes`, storing it in `diet_coef_no_interaction`. 

In [ ]:
hr_model_no_interaction <- NULL # YOUR CODE HERE
diet_coef_no_interaction <- NULL # YOUR CODE HERE

# Check your fitted model
summary(hr_model_no_interaction)

In [ ]:
. = ottr::check("tests/q3_1.R")

**Question 3.2.** In the no-interaction model, the coefficient for `on_dietyes` corresponds to the estimated difference in resting heart rate between dieting and non-dieting adults, holding the average number of daily steps constant. Which of the following correctly describes what this model assumes about the relationship between steps and heart rate? 

1. The effect of each additional step is assumed to be zero for the dieting group and non-zero for the non-dieting group.
2. The effect of each additional step is assumed to be the same for both diet groups (i.e., the regression lines are parallel).
3. The dieting group is assumed to have a higher intercept but the same slope as the non-dieting group.
4. The model makes no assumption about the relationship between the average number of daily steps and the resting heart rate.

Assign your answer (1, 2, 3, or 4) to `no_interaction_assumption`. 

In [ ]:
no_interaction_assumption <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_2.R")

**Question 3.3.** Now fit the full interaction model and store it in `hr_model_with_interaction`. Extract the interaction coefficient and store it in `hr_interaction_coef`.

In [ ]:
hr_model_with_interaction <- NULL # YOUR CODE HERE
hr_interaction_coef <- NULL # YOUR CODE HERE

# Check your fitted model
summary(hr_model_with_interaction)

In [ ]:
. = ottr::check("tests/q3_3.R")

**Question 3.4.** Using the coefficients from `hr_model_with_interaction`, compute the estimated effect of *an additional 1,000 daily steps* on the resting heart rate for each diet group separately. Store these as a numeric vector `thousand_step_effects` with names `"no_diet"` and `"on_diet"` (in that order). 

Recall that in the interaction model (with no errors), we have 

$$ \widehat{y}_i = \widehat{\beta}_0 + \widehat{\beta}_1x_{1i} + \widehat{\beta}_2x_{2i} + \widehat{\beta}_3 x_{1i}x_{2i}, $$

where $x_{1i}$ denotes the average number of steps subject $i$ walked daily, $x_{2i}$ denotes whether subject $i$ is on a diet (0 = "no", 1 = "yes"), and $x_{1i}x_{2i}$ is the interaction term. For the non-dieting group, the slope is $\widehat{\beta}_1$, whereas it is $\widehat{\beta}_1 + \widehat{\beta}_3$ for the dieting group. Be sure to multiply both slopes by 1,000 to express this new effect per 1,000 steps. 

*Note: It may be helpful to store your intermediate steps in their own variables, as is suggested in the code.*

In [ ]:
# Extract coefficients
# b1 <- ....
# b3 <- ....

# Calculate effect of 1k additional steps on both groups
thousand_step_effects <- NULL # YOUR CODE HERE
thousand_step_effects

In [ ]:
. = ottr::check("tests/q3_4.R")

**Question 3.5.** Run a partial $F$-test to assess whether `on_diet` (including its interaction with `daily_steps`) contributes to meaningfully explaining heart rate; that is, test $H_0: \beta_2 = \beta_3 = 0$. Compare `hr_model_with_interaction` to a null model `hr_null_model` containing only `daily_steps`. Store the obtained $p$-value of this comparison in `pval_diet_joint`. 

In [ ]:
hr_null_model <- NULL # YOUR CODE HERE
pval_diet_joint <- NULL # YOUR CODE HERE


# Summary of null model
summary(hr_null_model)

In [ ]:
. = ottr::check("tests/q3_5.R")

**Question 3.6.1.** In the interaction plot for this heart rate study, the two regressions lines are not parallel. This implies that there exists a point at which they might intersect; we could calculate the specific average number of daily steps at which the predicted heart rate for both the dieting group and the non-dieting group should be the same. If this equivalence point is denoted by $\tilde{x}_1$, which is the correct expression for $\tilde{x}_1$? 

1. $\tilde{x}_1 = -\frac{\widehat{\beta}_2}{\widehat{\beta}_3}.$
2. $\tilde{x}_1 = \frac{\widehat{\beta}_1}{\widehat{\beta}_3}.$
3. $\tilde{x}_1 = -\frac{\widehat{\beta}_0 + \widehat{\beta}_2}{\widehat{\beta}_1 + \widehat{\beta}_3}.$
4. $\tilde{x}_1 = \frac{\widehat{\beta}_2}{\widehat{\beta}_3}.$

Assign your answer (1, 2, 3, or 4) to `steps_equivalence_exp`. 

*Hint: Let $\widehat{y}_d = \widehat{\beta}_0 + \widehat{\beta}_1x_{1} + \widehat{\beta}_2 + \widehat{\beta}_3x_1$ denote the model for the dieting group (take the original, full model and set $x_2 = 1$), and $\widehat{y}_{nd} = \widehat{\beta}_0 + \widehat{\beta}_1x_{1}$ denote the model for the non-dieting group (take the original, full model and set $x_2 = 0$). The equivalence point should be the value for $x_1$, which corresponds to the daily average number of steps, when the two models are the same.*

In [ ]:
steps_equivalence_exp <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_6_1.R")

**Question 3.6.2.** The previous question hinted at the possible existence of a point of intersection in the interaction model because the regression lines are not parallel to each other. More specifically, the estimated slope for the average daily steps is more negative (steeper) for the dieting group than for the non-dieting group. Which statement best explains the contextual interpretation behind this difference in the slopes? 

1. People on a diet are more likely to walk more steps on average per day, which explains why their heart rate is lower.
2. Each additional step is estimated to be associated with a larger decrease in resting heart rate for those who diet than for those who do not, suggesting exercise and diet may jointly contribute to better health. 
3. Dieting has no effect on a person's resting heart rate because the interaction $p$-value is very large.
4. The association between dieting and resting heart rate is consistent across all step counts, meaning the benefit of dieting is the same regardless of how many steps a person takes daily on average.

Assign your answer (1, 2, 3, or 4) to `diff_slopes_interp` below. 

In [ ]:
diff_slopes_interp <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_6_2.R")

**Question 3.7.** At any given step count $x_1$, the estimated difference in the resting heart rate between the dieting and the non-dieting groups is $\widehat{D}(x_1) = \widehat{\beta}_2 + \widehat{\beta}_3x_1$. Using the variance-covariance matrix of the fully fitted model, calculate the standard error of this difference specifically at $x_1 = 10,000$ steps. Store this result in `se_diff_10k`. 

**Note: You will need to make use of the coefficient names of `hr_model_with_interaction`, which should include `"(Intercept)"`, `"daily_steps"`, `"on_dietyes"`, and `"daily_steps:on_dietyes"`. Using the variance formula in *Hint \#2*, you will need to figure out which coefficients you will need to use. Some starter code has been provided in the comments.**

*Hint \#1: Given a model `model`, the variance-covariance matrix is given by `vcov(model)`.*

*Hint \#2: Assuming $c$ is a fixed value (constant), the general formula for the variance of a sum is $$\mathrm{Var}\big(\alpha + c\beta\big) = \mathrm{Var}(\alpha) + c^2\mathrm{Var}(\beta) + 2c\mathrm{Cov}[\alpha, \beta].$$ Plug in the right coefficients for $\alpha$ and $\beta$.*

In [ ]:
# Extract variance-covariance matrix
num_steps <- 10000
vcov_matrix <- NULL # YOUR CODE HERE

# Apply variance formula from hint 
# ....

# Calculate standard error 
se_diff_10k <- NULL # YOUR CODE HERE
se_diff_10k


In [ ]:
. = ottr::check("tests/q3_7.R")

**Question 3.8.** Sometimes, it does not make sense to rely on certain model assumptions, like the normality of the errors. Assumptions being violated could affect our final model and its coefficients. We can use bootstrapping to try and estimate how much the slope difference between the dieting and the non-dieting groups actually varies. 

Fill in the code below to: 
1. Resample `heart_df` with replacement 1000 times. 
2. For each iteration, fit the interaction model and record the interaction coefficient $\widehat{\beta}_3$. This will generate a distribution for the value of the coefficient.
3. Find the 95\% confidence interval for the interaction coefficient $\widehat{\beta}_3$ and store the resulting confidence interval as `interaction_boot_ci`. 

In [ ]:
set.seed(123) # DO NOT DELETE OR CHANGE THIS LINE!
n_boot <- NULL # YOUR CODE HERE

# Perform bootstrapping
boot_dist <- replicate(n = n_boot, exp = {
    boot_indices <- NULL # YOUR CODE HERE
    boot_sample <- NULL # YOUR CODE HERE
    boot_model <- NULL # YOUR CODE HERE
    boot_int_coef <- NULL # YOUR CODE HERE
})

# Calculate 95% confidence interval
interaction_boot_ci <- NULL # YOUR CODE HERE
interaction_boot_ci



In [ ]:
. = ottr::check("tests/q3_8.R")

Any of the fitted linear models we have looked at for this heart rate study assume that the errors are normally distributed, or that the residuals are random with mean 0. Run the following cell to see how well our residuals follow this assumption of randomness.

In [ ]:
# Extract values
hr_fitted <- predict(hr_model_with_interaction)
hr_residuals <- residuals(hr_model_with_interaction)

# Plot residuals vs fitted values
res_df <- data.frame(fitted = hr_fitted, resid = hr_residuals)
ggplot(res_df, aes(x = fitted, y = resid)) +
  geom_point(alpha = 0.5) +
  geom_hline(aes(yintercept = 0, color = "Zero Line"), linetype = "solid") +
  geom_hline(aes(yintercept = mean(resid), color = "Mean of Residuals"), linetype = "solid") +
  labs(title = "Residuals vs. Fitted", x = "Fitted Values", y = "Residuals", color = "Reference Lines") + 
  scale_color_manual(values = c("Zero Line" = "red", "Mean of Residuals" = "blue")) +
  theme_minimal()


This scatterplot shows that the residuals do not take a particular shape (e.g. U-shaped or some other linear shape) and are roughly distributed with mean 0.

## Congratulations, you are finished!!

To submit your assignment:

1. Select `Kernel -> Restart Kernel and Run All Cells...` to ensure that you have executed all cells, including the test cells.
2. Read through the notebook to make sure everything is fine and all tests passed.
3. Download your notebook using `File -> Download`, then upload your notebook to Gradescope.
4. Stick around while the Gradescope autograder grades your work. Make sure you see that all tests have passed on Gradescope.
5. Check that you have a confirmation email from Gradescope and save it as proof of your submission.